# HAR — Dataset Exploration
Use this notebook to explore the HMDB51 dataset, visualise frame samples, and sanity-check the data pipeline before training.

In [ ]:
import sys
sys.path.insert(0, '..')   # so we can import from the project root

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import torch

import config
from utils.dataset import HMDB51Dataset, get_class_labels, sample_frames

## 1. Class distribution

In [ ]:
label_map = get_class_labels(config.SPLITS_DIR)
class_names = sorted(label_map.keys())
print(f'Number of classes: {len(class_names)}')
print('Classes:', class_names)

In [ ]:
# Count samples per class from split files
import glob as _glob
class_counts = {}
for cls in class_names:
    split_file = os.path.join(config.SPLITS_DIR, f'{cls}_test_split{config.SPLIT_ID}.txt')
    if not os.path.exists(split_file):
        continue
    with open(split_file) as f:
        lines = [l.strip().split() for l in f if l.strip()]
    class_counts[cls] = sum(1 for _, code in lines if code in ('1', '2'))

sorted_counts = dict(sorted(class_counts.items(), key=lambda x: -x[1]))

fig, ax = plt.subplots(figsize=(14, 8))
ax.bar(sorted_counts.keys(), sorted_counts.values(), color='steelblue')
ax.set_xticklabels(sorted_counts.keys(), rotation=90, fontsize=7)
ax.set_ylabel('Video clips')
ax.set_title('HMDB51 — Samples per class')
plt.tight_layout()
plt.show()

## 2. Sample frame sequences

In [ ]:
# Pick 3 random videos and show their sampled frames
video_files = glob.glob(os.path.join(config.DATA_DIR, '**', '*.avi'), recursive=True)
rng = np.random.default_rng(42)
sample_vids = rng.choice(video_files, size=min(3, len(video_files)), replace=False)

for vid_path in sample_vids:
    class_name = os.path.basename(os.path.dirname(vid_path))
    frames = sample_frames(vid_path, num_frames=8)
    if frames is None:
        continue
    fig, axes = plt.subplots(1, 8, figsize=(16, 2))
    fig.suptitle(f'Class: {class_name}  |  {os.path.basename(vid_path)}', fontsize=9)
    for ax, frame in zip(axes, frames):
        ax.imshow(frame)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 3. Dataset loader sanity check

In [ ]:
ds = HMDB51Dataset(
    data_dir=config.DATA_DIR,
    splits_dir=config.SPLITS_DIR,
    split='train',
    split_id=config.SPLIT_ID,
    num_frames=8,
    debug=True,
    debug_samples=20,
)

sample = ds[0]
print('frames shape:', sample['frames'].shape)   # (T, C, H, W)
print('label:',        sample['label'].item())
print('class:',        sample['class_'])

## 4. Model forward pass check

In [ ]:
from models.cnn_lstm import CNNLSTM

model = CNNLSTM(num_classes=config.NUM_CLASSES)
stats = model.count_parameters()
print(f'Total params:     {stats["total"]:,}')
print(f'Trainable params: {stats["trainable"]:,}')

dummy = torch.randn(2, 8, 3, 224, 224)
out = model(dummy)
print(f'Output shape: {out.shape}')   # (2, 51)